In [1]:
import pandas as pd
import numpy as np
import zipfile
from datetime import datetime
import os
pd.set_option("display.max_colwidth", None)



In [2]:
espacios_col = [(0, 3), (3, 23), (23, 43), (43,45), (45, 48), (48, 49), (368, 376), (376, 384), (384, 392), (346, 347), (396, 411), (411, 426)
                ]

column_names = ["Tipo de seguro", "Certificado", "Numero Interno Del Canal", "Tipo de Registro", "Moneda", 
                "Tipo de Movimiento", "Fecha de Afiliacion", "Fecha de inicio del seguro", "Fecha fin del seguro",
                "Periodo de pago", "Monto Asegurado", "Prima"
                ]

In [3]:
def Extraer_fechas(filename):
    try:
        base = os.path.splitext(os.path.basename(filename))[0]
        # Primera fecha (ddmmyy → fecha completa)
        date_str = base[3:9]   # "030120"
        fecha_trama = datetime.strptime(date_str, "%d%m%y").date()

        # Segunda fecha (ddmm → usar año de la primera fecha)
        decl_str = base[10:14]  # "0601"
        dia, mes = int(decl_str[:2]), int(decl_str[2:])
        fecha_declarada = datetime(fecha_trama.year, mes, dia).date()

        return fecha_trama, fecha_declarada
    except Exception:
        return None, None  # Si algo falla


In [4]:
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [5]:
# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama Original"] = linea.rstrip("\n")       
    return row

In [ ]:
archivo_zip = []
with zipfile.ZipFile('C:/data/TRAMAS/RED_2020-01.zip', "r") as archivo_zip:
    tramas_txt = archivo_zip.namelist()
    #for file in tramas_txt:
        #with archivo_zip.open(file) as file_txt:
        #    file_content = file_txt.read()
        #    list_lines = file_content.decode('latin-1').splitlines()
    file_content= archivo_zip.read(tramas_txt[2])
    lista_lineas = file_content.decode('latin-1').splitlines()
    print(lista_lineas[0:3])
print(tramas_txt)


['703001101013140001502280101                00PEN4                                                                                                                                                                                                                                                                                                         M                     202001062019122720200127000{00000000600000{00000000000480I                                                                                                                                                                                                                                                                                          ', '703001101021840002808010102                00PEN4                                                                                                                                                                                                                                             

In [6]:
#with open("C:/data/M_SUSTCRIS100_DESGRAVAMEN.txt", "r", encoding="latin-1") as file:
with open("C:/data/RED030226_0402.TXT", "r", encoding="latin-1") as file:
    lista_lineas =  file.readlines()

In [7]:
df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])

In [8]:
df_tramas.head(3)

,Tipo de seguro,Certificado,Numero Interno Del Canal,Tipo de Registro,Moneda,Tipo de Movimiento,Fecha de Afiliacion,Fecha de inicio del seguro,Fecha fin del seguro,Periodo de pago,Monto Asegurado,Prima,Trama Original
0,703,00110108874000131779,0108,00,PEN,4,20260203,20260201,20260301,M,00000000600000{,00000000000480I,703001101088740001317790108 00PEN4 M 202602032026020120260301000{00000000600000{00000000000480I
1,703,00110121114000088769,0114,00,PEN,4,20260203,20260203,20260303,M,00000000600000{,00000000000754D,703001101211140000887690114 00PEN4 M 202602032026020320260303000{00000000600000{00000000000754D
2,703,00110139304000214061,0139,00,PEN,4,20260203,20260202,20260302,M,00000000600000{,00000000000480I,703001101393040002140610139 00PEN4 M 202602032026020220260302000{00000000600000{00000000000480I


In [9]:
letra = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
valor = np.array(list("1234567891234567890000000000123456789"))
diccionario_reemplazo = dict(zip(letra, valor))
def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan
        
        parte_numerica = valor_str[:-1]
        letra_final = valor_str[-1]
        # Buscar equivalencia
        digito = diccionario_reemplazo.get(letra_final)
        if digito is None:
            return np.nan  # letra desconocida → nulo
        # Construir número completo
        numero_str = parte_numerica + digito
        # Intentar convertir a Decimal
        return float(numero_str) / 100

    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo


In [10]:
df_tramas["Monto Asegurado"] = df_tramas["Monto Asegurado"].apply(convertir_prima)
df_tramas["Prima Bruta"] = df_tramas["Prima"].apply(convertir_prima)
df_tramas["Error Prima"] = df_tramas["Prima Bruta"].isna()

In [11]:
df_tramas["Fecha de Afiliacion"] = Convertir_fecha(df_tramas["Fecha de Afiliacion"])
df_tramas["Fecha de inicio del seguro"] = Convertir_fecha(df_tramas["Fecha de inicio del seguro"])
df_tramas["Fecha fin del seguro"] = Convertir_fecha(df_tramas["Fecha fin del seguro"])

In [12]:
#fecha_trama, fecha_declarada = Extraer_fechas(tramas_txt[2])
fecha_trama, fecha_declarada = Extraer_fechas('RED030226_0402.TXT')
df_tramas["Fecha Trama"] = fecha_trama
df_tramas["Fecha Declarada"] = fecha_declarada
df_tramas['Fecha Trama'] = pd.to_datetime(df_tramas['Fecha Trama'], errors='coerce')
df_tramas['Fecha Declarada'] = pd.to_datetime(df_tramas['Fecha Declarada'], errors='coerce')

In [13]:
df_tramas.columns = (df_tramas.columns
                     .str.strip()  # quitar espacios al inicio/fin
                     .str.upper()  # opcional: todo en mayúsculas
                     .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
)

In [27]:
df_tramas['LEN_TRAMA'] = df_tramas['TRAMA_ORIGINAL'].str.len()

In [28]:
df_tramas.head()

,TIPO_DE_SEGURO,CERTIFICADO,NUMERO_INTERNO_DEL_CANAL,TIPO_DE_REGISTRO,MONEDA,TIPO_DE_MOVIMIENTO,FECHA_DE_AFILIACION,FECHA_DE_INICIO_DEL_SEGURO,FECHA_FIN_DEL_SEGURO,PERIODO_DE_PAGO,MONTO_ASEGURADO,PRIMA,TRAMA_ORIGINAL,PRIMA_BRUTA,ERROR_PRIMA,FECHA_TRAMA,FECHA_DECLARADA,LEN_TRAMA
0,703,00110108874000131779,0108,00,PEN,4,2026-02-03,2026-02-01,2026-03-01,M,60000.0,00000000000480I,703001101088740001317790108 00PEN4 M 202602032026020120260301000{00000000600000{00000000000480I,48.09,False,2026-02-03,2026-02-04,708
1,703,00110121114000088769,0114,00,PEN,4,2026-02-03,2026-02-03,2026-03-03,M,60000.0,00000000000754D,703001101211140000887690114 00PEN4 M 202602032026020320260303000{00000000600000{00000000000754D,75.44,False,2026-02-03,2026-02-04,708
2,703,00110139304000214061,0139,00,PEN,4,2026-02-03,2026-02-02,2026-03-02,M,60000.0,00000000000480I,703001101393040002140610139 00PEN4 M 202602032026020220260302000{00000000600000{00000000000480I,48.09,False,2026-02-03,2026-02-04,708
3,703,00110178154000196333,0178,01,PEN,4,2026-02-03,2026-01-26,2026-02-26,M,60000.0,00000000000480I,703001101781540001963330178 01PEN4 M 202602032026012620260226000{00000000600000{00000000000480I,48.09,False,2026-02-03,2026-02-04,708
4,703,00110179924000160873,0387,00,PEN,4,2026-02-03,2026-02-03,2026-03-03,M,60000.0,00000000000480I,703001101799240001608730387 00PEN4 M 202602032026020320260303000{00000000600000{00000000000480I,48.09,False,2026-02-03,2026-02-04,708


In [30]:
df_tramas['ERROR_PRIMA'].value_counts()

ERROR_PRIMA
False    21102
True      2743
Name: count, dtype: int64

In [31]:
df_tramas[df_tramas['ERROR_PRIMA'] == True]['TIPO_DE_MOVIMIENTO'].value_counts()

TIPO_DE_MOVIMIENTO
0    1593
5    1109
9      16
7       8
2       6
1       3
4       3
8       2
3       2
G       1
Name: count, dtype: int64

In [32]:
df_tramas[(df_tramas['ERROR_PRIMA'] == True) & (df_tramas['TIPO_DE_MOVIMIENTO'] == '4')].head()

,TIPO_DE_SEGURO,CERTIFICADO,NUMERO_INTERNO_DEL_CANAL,TIPO_DE_REGISTRO,MONEDA,TIPO_DE_MOVIMIENTO,FECHA_DE_AFILIACION,FECHA_DE_INICIO_DEL_SEGURO,FECHA_FIN_DEL_SEGURO,PERIODO_DE_PAGO,MONTO_ASEGURADO,PRIMA,TRAMA_ORIGINAL,PRIMA_BRUTA,ERROR_PRIMA,FECHA_TRAMA,FECHA_DECLARADA,LEN_TRAMA
4687,902,00110136934000282591,00110136979600169693,03,PEN,4,NaT,NaT,NaT,,NaN,,902001101369340002825910011013697960016969303PEN4152418 L45425712 PEN000000000101A20260203I1 I1 00000000000000{00000000000101A,NaN,True,2026-02-03,2026-02-04,708
4689,902,00110133624000411131,00110133699600219376,03,PEN,4,NaT,NaT,NaT,,NaN,,902001101336240004111310011013369960021937603PEN4113820 L44609332 PEN000000000189G20260203I1 I1 00000000000000{00000000000189G,NaN,True,2026-02-03,2026-02-04,708
19391,959,00110304394000732360,00110304369600349015,03,PEN,4,NaT,NaT,NaT,,NaN,,959001103043940007323600011030436960034901503PEN4150111 L41650484 PEN000000000614D00010101I1 I1 00000000000000{00000000000614D,NaN,True,2026-02-03,2026-02-04,708


In [33]:
df_tramas['LEN_TRAMA'].value_counts()

LEN_TRAMA
708    23845
Name: count, dtype: int64

In [18]:
df_filtrado = df_tramas[df_tramas["PRIMA_BRUTA"].isna()]
df_filtrado.head(3)

,TIPO_DE_SEGURO,CERTIFICADO,NUMERO_INTERNO_DEL_CANAL,TIPO_DE_REGISTRO,MONEDA,TIPO_DE_MOVIMIENTO,FECHA_DE_AFILIACION,FECHA_DE_INICIO_DEL_SEGURO,FECHA_FIN_DEL_SEGURO,PERIODO_DE_PAGO,PRIMA,TRAMA_ORIGINAL,PRIMA_BRUTA,ERROR_PRIMA


In [19]:
df_tramas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4206 entries, 0 to 4205
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   TIPO_DE_SEGURO              4206 non-null   object        
 1   CERTIFICADO                 4206 non-null   object        
 2   NUMERO_INTERNO_DEL_CANAL    4206 non-null   object        
 3   TIPO_DE_REGISTRO            4206 non-null   object        
 4   MONEDA                      4206 non-null   object        
 5   TIPO_DE_MOVIMIENTO          4206 non-null   object        
 6   FECHA_DE_AFILIACION         0 non-null      datetime64[ns]
 7   FECHA_DE_INICIO_DEL_SEGURO  4206 non-null   datetime64[ns]
 8   FECHA_FIN_DEL_SEGURO        4206 non-null   datetime64[ns]
 9   PERIODO_DE_PAGO             4206 non-null   object        
 10  PRIMA                       4206 non-null   object        
 11  TRAMA_ORIGINAL              4206 non-null   object      